##### DataFrames

A DataFrame is the most common Structured API and simply represents a table of data with
rows and columns. The list that defines the columns and the types within those columns is called
the schema. 

The reason for putting
the data on more than one computer should be intuitive: either the data is too large to fit on one
machine or it would simply take too long to perform that computation on one machine.

#### Partitions

To allow every executor to perform work in parallel, Spark breaks up the data into chunks called
partitions. 


 If you have one partition, Spark will have a parallelism of only one,
even if you have thousands of executors. If you have many partitions but only one executor,
Spark will still have a parallelism of only one because there is only one computation resource.


##### Transformations


In Spark, the core data structures are immutable, meaning they cannot be changed after they’re
created. This might seem like a strange concept at first: if you cannot change it, how are you
supposed to use it? 


you need to instruct Spark how you would like to
modify it to do what you want. These instructions are called transformations.

In [2]:
from pyspark.sql.session import SparkSession

def get_spark_session(appName="default"):
    spark = (
        SparkSession
        .builder
        .appName(appName)
        .master("local[*]")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("ERROR")
    return spark

def stop_start(app_name):
   
    global spark 
    
    if 'spark' in globals():
        try:
            spark.stop()
            print("Old Spark session successfully stopped.")
        except Exception as e:
             print(f"No active session to stop. Error: {e}")
    else:
        print("No existing 'spark' variable found.")

    # 3. Return the fresh session
    print("Starting new Spark session...")
    return get_spark_session(app_name)

In [3]:
spark = stop_start()

TypeError: stop_start() missing 1 required positional argument: 'app_name'

In [ ]:
myRange = spark.range(1000).toDF("number")

divisBy2 = myRange.where("number % 2 = 0")

##### Notice that above code return no output. This is because we specified only an abstract transformation

Spark will not act on transformations until we call an action

**Transformations** are the core of how you express your business logic using Spark. There are two
types of transformations: those that specify narrow dependencies, and those that specify wide
dependencies.

##### **Narrow dependicies**

Transformations consisting of narrow dependencies (we’ll call them narrow transformations) are
those for which each input partition will contribute to only one output partition.


 In the preceding
code snippet, the where statement specifies a narrow dependency, where only one partition
contributes to at most one output partition,


<img src="../data/pictures/narrow_dependecies.png" width="400" />

### Wide dependencies or Wide Transformation

A wide dependency (or wide transformation) style transformation will have input partitions
contributing to many output partitions. You will often hear this referred to as a shuffle whereby
Spark will exchange partitions across the cluster

<img src="../data/pictures/wide_dependencies.png" width="400" />

### With narrow transformations, Spark will
automatically perform an operation called pipelining, meaning that if we specify multiple filters
on DataFrames, they’ll all be performed in-memory. The same cannot be said for shuffles. When
we perform a shuffle, Spark writes the results to disk.

### Lazy Evaluation
Spark will wait until the very last moment to execute the graph of
computation instructions.


you build up a plan of transformations that you would like to apply to your
source data. 

By waiting until the last minute to execute the code, Spark compiles this plan from
your raw DataFrame transformations to a streamlined physical plan that will run as efficiently as
possible across the cluster. 


### Benifits of the lazy evaluation

- optimization before actual execution happen over cluster

what if you want only one record , but you reading whole data on the s3


### Actions

Transformations allow us to build up our logical transformation plan.
To trigger the computation, we run an action.

An action instructs Spark to compute a result from a series of transformations.

- Actions to view data in the console
- Actions to collect data to native objects in the respective language
- Actions to write to output data sources

In [ ]:
divisBy2.count()

500

In [ ]:
divisBy2.explain()

== Physical Plan ==
*(1) Project [id#0L AS number#1L]
+- *(1) Filter ((id#0L % 2) = 0)
   +- *(1) Range (0, 1000, step=1, splits=8)




In [ ]:
! head ../data/flight-data/csv/2015-summary.csv

DEST_COUNTRY_NAME,ORIGIN_COUNTRY_NAME,count
United States,Romania,15
United States,Croatia,1
United States,Ireland,344
Egypt,United States,15
United States,India,62
United States,Singapore,1
United States,Grenada,62
Costa Rica,United States,588
Senegal,United States,40


In [ ]:
flightData2015 = (
    spark
    .read
    .option("header", "true")
    .option("inferSchema","true")
    .csv("../data/flight-data/csv/2015-summary.csv")
)

flightData2015.take(5)

[Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='Romania', count=15),
 Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='Croatia', count=1),
 Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='Ireland', count=344),
 Row(DEST_COUNTRY_NAME='Egypt', ORIGIN_COUNTRY_NAME='United States', count=15),
 Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='India', count=62)]

In [ ]:
spark.stop()

In [ ]:

spark = stop_start("cell-10")

flightData2015 = (
    spark
    .read
    .option("header", "true")
    .csv("../data/flight-data/csv/2015-summary.csv")
)


flightData2015.write.mode("overwrite").parquet("../data/output/flight-data-2025")

Old Spark session successfully stopped.
Starting new Spark session...


In [ ]:
## sort the data
spark = stop_start("cell-11")

flightData2015 = (
    spark
    .read
    .option("header", "true")
    .csv("../data/flight-data/csv/2015-summary.csv")
)


flightData2015.sort("count").explain()

Old Spark session successfully stopped.
Starting new Spark session...
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [count#137 ASC NULLS FIRST], true, 0
   +- Exchange rangepartitioning(count#137 ASC NULLS FIRST, 200), ENSURE_REQUIREMENTS, [plan_id=213]
      +- FileScan csv [DEST_COUNTRY_NAME#135,ORIGIN_COUNTRY_NAME#136,count#137] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/suanki/Desktop/projects/suanki-pyspark-and-optimization/dat..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<DEST_COUNTRY_NAME:string,ORIGIN_COUNTRY_NAME:string,count:string>




In [ ]:
## sort the data
execution_cell = get_ipython().execution_count
spark = stop_start(f"cell-{execution_cell}")

spark.conf.set("spark.sql.shuffle.partitons", "5")

flightData2015 = (
    spark
    .read
    .option("inferSchema","true")
    .option("header", "true")
    .csv("../data/flight-data/csv/2015-summary.csv")
)


flightData2015.sort("count").take(2)

Old Spark session successfully stopped.
Starting new Spark session...


[Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='Singapore', count=1),
 Row(DEST_COUNTRY_NAME='Moldova', ORIGIN_COUNTRY_NAME='United States', count=1)]

### lineage

The logical plan of transformations that we build up defines a lineage for the DataFrame so that
at any given point in time, Spark knows how to recompute any partition by performing all of the
operations it had before on the same input data. 


- We do not manipulate the physical data, instead we configure the pgysical execution characteristics things like shuffle partions parameter

In [ ]:
### DataFrame and SQL

flightData2015.createOrReplaceTempView("flight_data_2015")

In [ ]:
sqlWay = spark.sql(""" 
select dest_country_name, count(1)
                   from flight_data_2015
                   group by dest_country_name
""")


dataFrameWay = (
    flightData2015
.groupBy("dest_country_name")
.count()
)

sqlWay.explain()

dataFrameWay.explain()


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[dest_country_name#180], functions=[count(1)])
   +- Exchange hashpartitioning(dest_country_name#180, 200), ENSURE_REQUIREMENTS, [plan_id=321]
      +- HashAggregate(keys=[dest_country_name#180], functions=[partial_count(1)])
         +- FileScan csv [DEST_COUNTRY_NAME#180] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/suanki/Desktop/projects/suanki-pyspark-and-optimization/dat..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<DEST_COUNTRY_NAME:string>


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[dest_country_name#180], functions=[count(1)])
   +- Exchange hashpartitioning(dest_country_name#180, 200), ENSURE_REQUIREMENTS, [plan_id=334]
      +- HashAggregate(keys=[dest_country_name#180], functions=[partial_count(1)])
         +- FileScan csv [DEST_COUNTRY_NAME#180] Batched: false, DataFilters: [], Format: CSV, Lo

In [ ]:
from pyspark.sql.functions import max

flightData2015.select(max("count")).take(1)

[Row(max(count)='986')]

In [ ]:
## find the top five destination countries in the data

In [ ]:
maxSql = spark.sql(""" 
  
select dest_country_name , sum(count) as destination_total
                   from flight_data_2015
                       group by dest_country_name
                       order by sum(count) desc
                   limit 5
""")


maxSql.show()



+-----------------+-----------------+
|dest_country_name|destination_total|
+-----------------+-----------------+
|    United States|         411352.0|
|           Canada|           8399.0|
|           Mexico|           7140.0|
|   United Kingdom|           2025.0|
|            Japan|           1548.0|
+-----------------+-----------------+



In [ ]:
## dataFrame syntax

from pyspark.sql.functions import desc,sum

flightDataGroupedDF = (flightData2015
 .groupBy("dest_country_name")
 .sum("count")
 .withColumnRenamed("sum(count)", "destination_total")
 .sort(desc("destination_total"))
 .limit(5)
 
)


flightDataGroupedDF.show()

+-----------------+-----------------+
|dest_country_name|destination_total|
+-----------------+-----------------+
|    United States|           411352|
|           Canada|             8399|
|           Mexico|             7140|
|   United Kingdom|             2025|
|            Japan|             1548|
+-----------------+-----------------+



In [ ]:
flightDataGroupedDF.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- TakeOrderedAndProject(limit=5, orderBy=[destination_total#357L DESC NULLS LAST], output=[dest_country_name#288,destination_total#357L])
   +- HashAggregate(keys=[dest_country_name#288], functions=[sum(count#290)])
      +- Exchange hashpartitioning(dest_country_name#288, 200), ENSURE_REQUIREMENTS, [plan_id=742]
         +- HashAggregate(keys=[dest_country_name#288], functions=[partial_sum(count#290)])
            +- FileScan csv [DEST_COUNTRY_NAME#288,count#290] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/suanki/Desktop/projects/suanki-pyspark-and-optimization/dat..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<DEST_COUNTRY_NAME:string,count:int>




### DAG

This execution plan is a directed acyclic graph (DAG) of transformations,
each resulting in a new immutable DataFrame, on which we call an action to generate a result

```
 You can also see how
our aggregation happens in two phases, in the partial_sum calls. This is because summing a list
of numbers is commutative, and Spark can perform the sum, partition by partition 
```

<img src="../data/pictures/transformation_flow_dag.png" width="600" />

### Datasets: Type-Safe Structured APIs

that DataFrames, which we just see above, are a distributed collection of
objects of type Row that can hold various types of tabular data. 

The Dataset API gives users the
ability to assign a Java/Scala class to the records within a DataFrame and manipulate it as a
collection of typed objects, similar to a Java ArrayList or Scala Seq. 

Datasets are type-safe, meaning that you cannot accidentally view the objects in a Dataset as
being of another class than the class you put in initially.

 This makes Datasets especially attractive for writing large applications,

 see *sparkAnalyticsRepo spark-with-scala module for dataset>
 <https://github.com/suankidev/sparkAnalytics/tree/main/spark-with-scala/src/main/scala/com/suanki/sparkwithScala>

The Dataset class is parameterized with the type of object contained inside: Dataset<T> in Java
and Dataset[T] in Scala. 

For example, a Dataset[Person] will be guaranteed to contain
objects of class Person.